# Preprocessing and Dataset Split

In [1]:
import pandas as pd

In [5]:
import boto3
import pandas as pd
import json


region = boto3.Session().region_name
s3_client = boto3.client("s3")

# project bucket
bucket_name = "aai-540-data"

# datasplit meta s3 folder
datasplit_folder = "dev_split"

## Step 1. Image Preprocessing and Feature Engineering
* Downselected original dataset to use on those with available bbox annotations
* Cropped images based on bbox coordinates
* Resized to 224x224 shape
* added 'location' column to provide spatial information (where the image was taken)
* added 'date_captured' column to provide temporal information

## Step 2. Split Dataset into Train-Val-Test-Prod (Time-Based)

In [3]:
# load master metadata from repo
grp_meta_df = pd.read_csv('../dataset/grp4_metadata_with_datetime.csv')
grp_meta_df.head()

,filename,label,category_id,bbox,image_id,location,split_type,date_captured
0,585f4d43-23d2-11e8-a6a3-ec086b02610b_0.jpg,cat,16,"[1200.64, 568.32, 146.7734375, 246.613359375]",585f4d43-23d2-11e8-a6a3-ec086b02610b,100,val,2010-05-25 20:32:56
1,59180801-23d2-11e8-a6a3-ec086b02610b_0.jpg,cat,16,"[999.253359375, 404.48, 129.706640625, 218.453...",59180801-23d2-11e8-a6a3-ec086b02610b,100,val,2010-05-25 20:32:57
2,588dba7d-23d2-11e8-a6a3-ec086b02610b_0.jpg,opossum,1,"[1213.44, 637.44, 212.48, 271.36]",588dba7d-23d2-11e8-a6a3-ec086b02610b,100,val,2010-05-25 22:35:28
3,58bea7d7-23d2-11e8-a6a3-ec086b02610b_0.jpg,opossum,1,"[1062.4, 808.96, 279.04, 238.08]",58bea7d7-23d2-11e8-a6a3-ec086b02610b,100,val,2010-05-25 22:35:29
4,58a8a344-23d2-11e8-a6a3-ec086b02610b_0.jpg,opossum,1,"[870.4, 960.0, 261.12, 337.92]",58a8a344-23d2-11e8-a6a3-ec086b02610b,100,val,2010-05-25 22:35:30


In [6]:
# Split Dataset as close as possible to 40-10-10-40 as required 

# Since we will do batch transforms monthly (in production set)
# 43% (28048) all images from the beginning til end of Dec 2011 - for training
# 9% (5914) all images from Jan 2012 to Feb 2012 - for hyperparameter tuning
# 12.2% (7932) all images from Mar 2012 to Apr 2012 - for final model performance
# 35.6% (23218) for production which means all images from May 2012 til end of dataset

# Load time-series list of images
df = grp_meta_df.copy() # replace code later using csv file generated by athena query
df = df.reset_index(drop=True)

# Define temporal splits
train_end = '2011-12-31'
val_end = '2012-02-31'
test_end = '2012-04-31'

# Create splits
train_df = df[df['date_captured'] <= train_end]
val_df = df[(df['date_captured'] > train_end) & (df['date_captured'] <= val_end)]
test_df = df[(df['date_captured'] > val_end) & (df['date_captured'] <= test_end)]
production_df = df[df['date_captured'] > test_end]

# On training set, remove labels with less than 10 image samples
# Then, remove those labels as well from the validation set
# No changes in test and production dfs, since they will simulate unseen data like new images and distribution shifts in the future

MIN_TRAIN_SAMPLES_PER_LABEL = 10

train_counts = train_df['label'].value_counts()
labels_to_keep = train_counts[train_counts >= MIN_TRAIN_SAMPLES_PER_LABEL].index
train_df = train_df[train_df['label'].isin(labels_to_keep)]
val_df = val_df[val_df['label'].isin(labels_to_keep)]
train_val_counts = pd.concat([train_df['label'].value_counts(), val_df['label'].value_counts()], axis=1).fillna(0).astype(int)
train_val_counts.columns = ['train_set', 'val_set']
display(train_val_counts)

# save to csv then upload to s3 bucket
train_df.to_csv('train-meta.csv', index=False, sep=',', header=True, encoding='utf-8')
val_df.to_csv('val-meta.csv', index=False, sep=',', header=True, encoding='utf-8')
test_df.to_csv('test-meta.csv', index=False, sep=',', header=True, encoding='utf-8')
production_df.to_csv('production-meta.csv', index=False, sep=',', header=True, encoding='utf-8')




,train_set,val_set
label,,
opossum,5730,1822
raccoon,5406,1052
rabbit,3034,271
bobcat,2979,454
coyote,2485,668
cat,1861,511
squirrel,1860,169
dog,1557,380
car,1547,424


In [7]:
# since dataset is arranged in time-series, labels mapping will be in increasing time order
# convert train/val labels to encoded integers
train_labels = train_df['label'].unique()
label_map = {label: idx for idx, label in enumerate((train_labels))}
display(label_map)


{'cat': 0,
 'opossum': 1,
 'squirrel': 2,
 'raccoon': 3,
 'bird': 4,
 'rabbit': 5,
 'dog': 6,
 'badger': 7,
 'bobcat': 8,
 'coyote': 9,
 'car': 10,
 'deer': 11,
 'rodent': 12,
 'skunk': 13,
 'empty': 14}

In [9]:
# upload to S3
s3_client.upload_file('train-meta.csv', bucket_name, f"{datasplit_folder}/train-meta.csv")
s3_client.upload_file('val-meta.csv', bucket_name, f"{datasplit_folder}/validation/val-meta.csv")
s3_client.upload_file('test-meta.csv', bucket_name, f"{datasplit_folder}/test/test-meta.csv")
s3_client.upload_file('production-meta.csv', bucket_name, f"{datasplit_folder}/production-meta.csv")

s3_client.put_object(
    Bucket=bucket_name,
    Key=f"{datasplit_folder}/label_mapping.json",
    Body=json.dumps(label_map)
)

{'ResponseMetadata': {'RequestId': 'HJVBM1NQ5CEA87J1',
  'HostId': 'BomAmzo6c42LiJEnXnRlPJ4g+UG01Pp5XwSJJOcVLL6sclNeRNaLDtmWyVlrVD5kzDVvHVJtpsR/Cpm3GgDEwm59GWceA8eAUQD99td1u0s=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'BomAmzo6c42LiJEnXnRlPJ4g+UG01Pp5XwSJJOcVLL6sclNeRNaLDtmWyVlrVD5kzDVvHVJtpsR/Cpm3GgDEwm59GWceA8eAUQD99td1u0s=',
   'x-amz-request-id': 'HJVBM1NQ5CEA87J1',
   'date': 'Mon, 23 Jun 2025 18:40:55 GMT',
   'x-amz-version-id': 'HNFRmaL85yl0dqwh7iyw6sJzqIt.9Ko.',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"9b1cda293454980bc31dd708f52bbe04"',
   'x-amz-checksum-crc32': 'lS8E2Q==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"9b1cda293454980bc31dd708f52bbe04"',
 'ChecksumCRC32': 'lS8E2Q==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256',
 'VersionId': 'HNFRmaL85yl0dqwh7iyw6sJzqIt.9Ko.'}

In [10]:
# generate train and val lst files for sagemaker model training
# add integer label mapping to train and val sets
train_df['label_enc'] = train_df['label'].map(label_map)
val_df['label_enc'] = val_df['label'].map(label_map)


train_df[['label_enc', 'filename']].to_csv('train.lst', sep='\t', header=False, index=True)
val_df[['label_enc', 'filename']].to_csv('validation.lst', sep='\t', header=False, index=True)


In [13]:
s3_client.upload_file('train.lst', bucket_name, f"{datasplit_folder}/train.lst")
s3_client.upload_file('validation.lst', bucket_name, f"{datasplit_folder}/validation/validation.lst")

In [14]:
len(train_df)

27906

In [15]:
len(val_df)

6048